# KonkaniVani ASR - Complete Retraining (FIXED)

## What's Fixed:
- ✅ CTC weight increased to 0.8 (was 0.3)
- ✅ Using full 88-hour dataset (was 21h)
- ✅ Periodic testing every 5 epochs
- ✅ Better learning rate and gradient clipping

## Expected Results:
- Epoch 20: Blank prob < 80% (model starts working)
- Epoch 40: Blank prob < 60% (good transcriptions)
- Epoch 100: Blank prob < 40% (production ready)

## Step 1: Setup Environment

In [1]:
# Install dependencies
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard matplotlib

# Suppress dependency warnings
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [2]:
import os
import sys
import json
import torch
import torchaudio
from pathlib import Path
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


## Step 2: Check Dataset

In [3]:
# List available datasets
!ls -lh /kaggle/input/

total 0
drwxr-xr-x 3 nobody nogroup 0 Dec 14 15:24 kaggle-training-scripts
drwxr-xr-x 3 nobody nogroup 0 Dec 14 15:24 konkani-asr-complete-data


In [4]:
# Set paths - UPDATE THESE to match your dataset names
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/konkani-asr-complete-data')  # Your main data
SCRIPTS_ROOT = Path('/kaggle/input/kaggle-training-scripts')  # Your scripts dataset

print(f"Data dataset: {DATA_ROOT}")
print(f"Scripts dataset: {SCRIPTS_ROOT}")
print("\nDataset structure:")
!ls -lh {DATA_ROOT}
print("\nScripts structure:")
!ls -lh {SCRIPTS_ROOT}

Data dataset: /kaggle/input/konkani-asr-complete-data
Scripts dataset: /kaggle/input/kaggle-training-scripts

Dataset structure:
total 0
drwxr-xr-x 3 nobody nogroup 0 Dec 14 15:24 konkani-10k

Scripts structure:
total 0
drwxr-xr-x 3 nobody nogroup 0 Dec 14 15:24 tmp


## Step 3: Extract and Prepare Data

In [5]:
# Copy training scripts from scripts dataset
import shutil
import zipfile

# Check if datasets exist
if not SCRIPTS_ROOT.exists():
    print(f"✗ ERROR: Scripts dataset not found at {SCRIPTS_ROOT}")
    print("\nPlease add the 'kaggle-training-scripts' dataset as input to this notebook.")
    print("Click 'Add Input' → Search for your scripts dataset → Add it")
    raise FileNotFoundError(f"Scripts dataset not found: {SCRIPTS_ROOT}")

if not DATA_ROOT.exists():
    print(f"✗ ERROR: Data dataset not found at {DATA_ROOT}")
    print("\nPlease add your data dataset as input to this notebook.")
    print("Click 'Add Input' → Search for your data dataset → Add it")
    print("\nAvailable datasets:")
    !ls -la /kaggle/input/
    raise FileNotFoundError(f"Data dataset not found: {DATA_ROOT}")

print("✓ Both datasets found!")
print("\nCopying training scripts...")
script_dirs = ['training_scripts', 'models', 'scripts', 'data']

for dir_name in script_dirs:
    src_dir = SCRIPTS_ROOT / 'tmp' / 'kaggle_scripts_package' / dir_name
    if src_dir.exists():
        dst_dir = Path('/kaggle/working') / dir_name
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"  ✓ Copied {dir_name}/")
    else:
        print(f"  ⚠️  {dir_name}/ not found at {src_dir}")

print("\n✓ Training scripts ready!")

# Extract data files if zipped
zip_files = list(DATA_ROOT.glob('*.zip'))
if zip_files:
    print(f"\nFound {len(zip_files)} data zip files. Extracting...")
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('/kaggle/working/')
    print("✓ Data extraction complete!")
else:
    print("\nNo data zip files found, copying data directly...")
    # Copy data files directly if not zipped
    for item in DATA_ROOT.iterdir():
        if item.is_dir():
            dst = Path('/kaggle/working') / item.name
            if not dst.exists():
                shutil.copytree(item, dst)
                print(f"  ✓ Copied {item.name}/")

✓ Both datasets found!

Copying training scripts...
  ✓ Copied training_scripts/
  ✓ Copied models/
  ✓ Copied scripts/
  ✓ Copied data/

✓ Training scripts ready!

No data zip files found, copying data directly...
  ✓ Copied konkani-10k/


In [6]:
# Verify extracted files
print("Checking extracted structure...")
print("\nPython scripts:")
!ls -lh /kaggle/working/training_scripts/*.py 2>/dev/null || echo "  ✗ training_scripts not found"
!ls -lh /kaggle/working/models/*.py 2>/dev/null || echo "  ✗ models not found"
!ls -lh /kaggle/working/scripts/*.py 2>/dev/null || echo "  ✗ scripts not found"

print("\nData files:")
!ls -lh /kaggle/working/ | head -15

Checking extracted structure...

Python scripts:
-rw-r--r-- 1 root root 21K Dec 14 15:24 /kaggle/working/training_scripts/train_konkanivani_asr.py
-rw-r--r-- 1 root root 11K Dec 14 15:24 /kaggle/working/models/konkanivani_asr.py
-rw-r--r-- 1 root root 8.1K Dec 14 15:24 /kaggle/working/scripts/generate_training_visualization.py
-rw-r--r-- 1 root root 8.1K Dec 14 15:24 /kaggle/working/scripts/test_best_model.py

Data files:
total 328K
drwxr-xr-x 3 root root 4.0K Dec 14 15:24 data
drwxr-xr-x 3 root root 4.0K Dec 14 15:24 konkani-10k
drwxr-xr-x 2 root root 4.0K Dec 14 15:24 models
---------- 1 root root 307K Dec 15 03:31 __notebook__.ipynb
drwxr-xr-x 2 root root 4.0K Dec 14 15:24 scripts
drwxr-xr-x 3 root root 4.0K Dec 14 15:24 training_scripts


## Step 4: Locate Dataset Manifests

In [7]:
# Find manifest files in the extracted data
import os
import json
# Search for manifest directories
possible_manifest_dirs = [
    '/kaggle/working/konkani-10k',  # Your dataset location
    '/kaggle/working/data/konkani-combined/manifests',
    '/kaggle/working/data/konkani-asr-v0/splits/manifests',
    '/kaggle/working/data/manifests'
]

manifest_dir = None
for dir_path in possible_manifest_dirs:
    # Check if directory exists and has manifest files
    if os.path.exists(dir_path):
        test_files = ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']
        if any(os.path.exists(os.path.join(dir_path, f)) for f in test_files):
            manifest_dir = Path(dir_path)
            print(f"✓ Found manifests at: {manifest_dir}")
            break

if manifest_dir is None:
    print("✗ No manifest directory found!")
    print("\nSearching for manifest files...")
    !find /kaggle/working -name "*manifest*.json" 2>/dev/null | grep -v "\._" | head -10

✓ Found manifests at: /kaggle/working/konkani-10k


In [8]:
# If manifests not found, try to prepare them
if manifest_dir is None or not manifest_dir.exists():
    print("Attempting to prepare manifests...")
    
    # Check if preparation script exists
    prep_script = Path('/kaggle/working/scripts/prepare_raw_corpus_data.py')
    if prep_script.exists():
        print("Running data preparation script...")
        !python /kaggle/working/scripts/prepare_raw_corpus_data.py
        manifest_dir = Path('/kaggle/working/data/konkani-combined/manifests')
    else:
        print("⚠️  Preparation script not found.")
        print("Please ensure your dataset includes pre-prepared manifest files.")
        print("\nExpected structure:")
        print("  data/manifests/train.json")
        print("  data/manifests/val.json")
        print("  data/manifests/test.json")

In [9]:
# Verify manifests and show dataset statistics
if manifest_dir and manifest_dir.exists():
    print("✓ Dataset manifests found:")
    print("=" * 60)
    
    total_duration = 0
    # Try both naming conventions
    manifest_files = [
        ('train_manifest.json', 'train.json'),
        ('val_manifest.json', 'val.json'),
        ('test_manifest.json', 'test.json')
    ]
    
    for primary_name, alt_name in manifest_files:
        manifest_path = manifest_dir / primary_name
        if not manifest_path.exists():
            manifest_path = manifest_dir / alt_name
        
        if manifest_path.exists():
            # Try loading as JSONL (one JSON per line) or JSON array
            data = []
            with open(manifest_path) as f:
                content = f.read().strip()
                try:
                    # Try as JSON array first
                    data = json.loads(content)
                except json.JSONDecodeError:
                    # Try as JSONL (one JSON object per line)
                    for line in content.split('\n'):
                        if line.strip():
                            data.append(json.loads(line))
            
            num_samples = len(data)
            
            # Calculate total duration if available
            duration = sum(item.get('duration', 0) for item in data)
            total_duration += duration
            
            display_name = manifest_path.name
            print(f"  {display_name:20s}: {num_samples:5,d} samples ({duration/3600:.1f}h)")
        else:
            print(f"  {primary_name:20s}: NOT FOUND")
    
    print("=" * 60)
    print(f"  Total Duration: {total_duration/3600:.1f} hours")
    print("✓ Ready to train!")
else:
    print("✗ ERROR: No manifest files found!")
    print("Cannot proceed with training.")

✓ Dataset manifests found:
  train_manifest.json : 7,215 samples (8.7h)
  val_manifest.json   :   896 samples (1.1h)
  test_manifest.json  :   889 samples (1.1h)
  Total Duration: 10.8 hours
✓ Ready to train!


## Step 5: Configure Training (FIXED SETTINGS)

In [10]:
# Training configuration with FIXES
import yaml

config = {
    'model': {
        'vocab_size': 82,
        'input_dim': 80,
        'd_model': 128,
        'encoder_layers': 8,
        'decoder_layers': 6,
        'num_heads': 4,
        'conv_kernel_size': 31,
        'dropout': 0.3
    },
    'training': {
        'learning_rate': 0.0001,      # 🔥 Increased from 0.0001
        'weight_decay': 0.0001,
        'grad_clip': 5.0,             # 🔥 Added gradient clipping
        'ctc_weight': 0.9,            # 🔥 CRITICAL FIX: was 0.3
        'batch_size': 8,
        'gradient_accumulation_steps': 2,
        'mixed_precision': True,
        'num_epochs': 100,            # 🔥 More epochs
        'save_every': 5,
        'test_every': 5               # 🔥 Test every 5 epochs
    },
    'data': {
        'train_manifest': str(manifest_dir / 'train_manifest.json') if (manifest_dir / 'train_manifest.json').exists() else str(manifest_dir / 'train.json'),
        'val_manifest': str(manifest_dir / 'val_manifest.json') if (manifest_dir / 'val_manifest.json').exists() else str(manifest_dir / 'val.json'),
        'vocab_file': str(manifest_dir / 'vocab.json') if (manifest_dir / 'vocab.json').exists() else '/kaggle/working/konkani-10k/vocab.json',
        'num_workers': 2
    },
    'paths': {
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'log_dir': '/kaggle/working/logs'
    },
    'device': 'cuda'
}

# Save config
os.makedirs('/kaggle/working/config', exist_ok=True)
with open('/kaggle/working/config/training_config_fixed.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Training config saved with FIXES:")
print(f"  - CTC weight: {config['training']['ctc_weight']} (was 0.3)")
print(f"  - Learning rate: {config['training']['learning_rate']} (was 0.0001)")
print(f"  - Gradient clip: {config['training']['grad_clip']} (was None)")
print(f"  - Testing: Every {config['training']['test_every']} epochs")

✓ Training config saved with FIXES:
  - CTC weight: 0.9 (was 0.3)
  - Learning rate: 0.0001 (was 0.0001)
  - Gradient clip: 5.0 (was None)
  - Testing: Every 5 epochs


In [11]:
# Fix audio paths in manifests to point to Kaggle locations
print("Fixing audio paths in manifests...")

for manifest_name in ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']:
    manifest_path = manifest_dir / manifest_name
    if not manifest_path.exists():
        manifest_path = manifest_dir / manifest_name.replace('_manifest', '')
    
    if manifest_path.exists():
        # Read manifest
        with open(manifest_path) as f:
            content = f.read().strip()
        
        # Parse as JSONL
        data = []
        for line in content.split('\n'):
            if line.strip():
                try:
                    data.append(json.loads(line))
                except:
                    pass
        
        # Fix paths
        fixed_count = 0
        for item in data:
            if 'audio_filepath' in item:
                old_path = item['audio_filepath']
                # Extract just the filename part after 'konkani-10k/audio/'
                if 'konkani-10k' in old_path or 'KonkaniRawSpeechCorpus' in old_path:
                    # Get the relative path from audio directory
                    if 'audio/' in old_path:
                        rel_path = old_path.split('audio/')[-1]
                    elif 'Data/' in old_path:
                        rel_path = old_path.split('Data/')[-1]
                    else:
                        rel_path = old_path.split('/')[-1]
                    
                    # Set new path
                    item['audio_filepath'] = f'/kaggle/working/konkani-10k/audio/Data/{rel_path}'
                    fixed_count += 1
        
        # Save fixed manifest
        with open(manifest_path, 'w') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"  ✓ {manifest_path.name}: Fixed {fixed_count}/{len(data)} paths")

print("\n✓ Manifest paths fixed!")

Fixing audio paths in manifests...
  ✓ train_manifest.json: Fixed 7215/7215 paths
  ✓ val_manifest.json: Fixed 896/896 paths
  ✓ test_manifest.json: Fixed 889/889 paths

✓ Manifest paths fixed!


## Step 6: Setup Python Path and Start Training

In [12]:
# Enable multi-GPU training with DataParallel
import torch

# Check GPU count
num_gpus = torch.cuda.device_count()
print(f"Using {num_gpus} GPU(s) for training")

if num_gpus > 1:
    print("✓ Multi-GPU training enabled!")
    print(f"  Batch size per GPU: {config['training']['batch_size']}")
    print(f"  Total batch per step: {config['training']['batch_size'] * num_gpus}")
    print(f"  Effective batch: {config['training']['batch_size'] * num_gpus * config['training']['gradient_accumulation_steps']}")


Using 2 GPU(s) for training
✓ Multi-GPU training enabled!
  Batch size per GPU: 8
  Total batch per step: 16
  Effective batch: 32


In [13]:
# Add working directory to Python path so imports work
import sys
sys.path.insert(0, '/kaggle/working')

print("Python path configured:")
print(f"  Working dir: /kaggle/working")
print(f"\nVerifying imports...")

try:
    from models.konkanivani_asr import create_konkanivani_model
    print("  ✓ models.konkanivani_asr")
except ImportError as e:
    print(f"  ✗ models.konkanivani_asr: {e}")

try:
    from data.audio_processing.audio_processor import AudioProcessor
    print("  ✓ data.audio_processing.audio_processor")
except ImportError as e:
    print(f"  ✗ data.audio_processing.audio_processor: {e}")

print("\n✓ Ready to train!")

Python path configured:
  Working dir: /kaggle/working

Verifying imports...
  ✓ models.konkanivani_asr
  ✓ data.audio_processing.audio_processor

✓ Ready to train!


In [14]:
# Step 6.5: Generate Custom Vocabulary from Training Data
print("🔧 Generating custom vocabulary from training data...")

import json
from collections import Counter
import os

def generate_custom_vocab_from_manifests(manifest_paths, min_freq=2):
    """Generate vocabulary from actual training data"""
    
    print("📊 Analyzing training data to build custom vocabulary...")
    
    # Collect all characters from training texts
    char_counter = Counter()
    total_samples = 0
    
    for manifest_path in manifest_paths:
        if not os.path.exists(manifest_path):
            print(f"⚠️  Manifest not found: {manifest_path}")
            continue
            
        print(f"  Processing: {os.path.basename(manifest_path)}")
        
        with open(manifest_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                try:
                    data = json.loads(line.strip())
                    text = data.get('text', '')
                    
                    # Count characters in this text
                    for char in text:
                        char_counter[char] += 1
                    
                    total_samples += 1
                    
                    if line_num % 1000 == 0:
                        print(f"    Processed {line_num} samples...")
                        
                except json.JSONDecodeError:
                    continue
    
    print(f"\n📈 Analysis complete:")
    print(f"  Total samples: {total_samples:,}")
    print(f"  Unique characters found: {len(char_counter):,}")
    
    # Filter characters by frequency
    filtered_chars = [char for char, freq in char_counter.items() if freq >= min_freq]
    print(f"  Characters with freq >= {min_freq}: {len(filtered_chars)}")
    
    # Essential tokens (always include)
    essential_tokens = ['<pad>', '<blank>', '<sos>', '<eos>', '<unk>']
    
    # Build vocabulary
    vocab_chars = essential_tokens + sorted(filtered_chars)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_vocab = []
    for char in vocab_chars:
        if char not in seen:
            unique_vocab.append(char)
            seen.add(char)
    
    # Create char2idx and idx2char mappings
    char2idx = {char: idx for idx, char in enumerate(unique_vocab)}
    idx2char = {idx: char for idx, char in enumerate(unique_vocab)}
    
    # Show character frequency stats
    print(f"\n📊 Character frequency analysis:")
    most_common = char_counter.most_common(20)
    for char, freq in most_common:
        display_char = repr(char) if char in [' ', '\n', '\t'] else char
        print(f"  {display_char:>8}: {freq:,}")
    
    return {
        'char2idx': char2idx,
        'idx2char': idx2char,
        'vocab_size': len(char2idx),
        'char_frequencies': dict(char_counter),
        'total_samples': total_samples
    }

# Generate custom vocabulary
manifest_files = []
for manifest_name in ['train_manifest.json', 'val_manifest.json', 'train.json', 'val.json']:
    manifest_path = manifest_dir / manifest_name
    if manifest_path.exists():
        manifest_files.append(str(manifest_path))

if manifest_files:
    print(f"Found {len(manifest_files)} manifest files:")
    for f in manifest_files:
        print(f"  - {f}")
    
    # Generate vocabulary with minimum frequency of 2
    custom_vocab = generate_custom_vocab_from_manifests(manifest_files, min_freq=2)
    
    # Save custom vocabulary
    custom_vocab_path = '/kaggle/working/custom_vocab.json'
    with open(custom_vocab_path, 'w', encoding='utf-8') as f:
        json.dump({
            'char2idx': custom_vocab['char2idx'],
            'idx2char': custom_vocab['idx2char'],
            'vocab_size': custom_vocab['vocab_size']
        }, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Custom vocabulary generated!")
    print(f"  Vocabulary size: {custom_vocab['vocab_size']} characters")
    print(f"  Saved to: {custom_vocab_path}")
    print(f"  Based on {custom_vocab['total_samples']:,} training samples")
    
    # Show sample of vocabulary
    print(f"\n📝 Sample vocabulary (first 20 characters):")
    for i, char in enumerate(list(custom_vocab['char2idx'].keys())[:20]):
        display_char = repr(char) if char in [' ', '\n', '\t'] else char
        print(f"  {i:2d}: {display_char}")
    
    # Update config to use custom vocabulary
    config['data']['vocab_file'] = custom_vocab_path
    config['model']['vocab_size'] = custom_vocab['vocab_size']
    
    print(f"\n🔧 Updated config:")
    print(f"  vocab_file: {config['data']['vocab_file']}")
    print(f"  vocab_size: {config['model']['vocab_size']}")
    
else:
    print("❌ No manifest files found! Cannot generate custom vocabulary.")
    print("Using fallback vocabulary...")


🔧 Generating custom vocabulary from training data...
Found 2 manifest files:
  - /kaggle/working/konkani-10k/train_manifest.json
  - /kaggle/working/konkani-10k/val_manifest.json
📊 Analyzing training data to build custom vocabulary...
  Processing: train_manifest.json
    Processed 1000 samples...
    Processed 2000 samples...
    Processed 3000 samples...
    Processed 4000 samples...
    Processed 5000 samples...
    Processed 6000 samples...
    Processed 7000 samples...
  Processing: val_manifest.json

📈 Analysis complete:
  Total samples: 8,111
  Unique characters found: 76
  Characters with freq >= 2: 76

📊 Character frequency analysis:
       ' ': 30,961
         ा: 27,706
         र: 11,155
         क: 10,656
         ्: 10,564
         ं: 10,041
         त: 9,548
         य: 8,479
         ल: 7,927
         े: 7,679
         ो: 7,499
         व: 7,312
         न: 7,256
         स: 7,252
         प: 5,917
         च: 5,775
         ी: 5,755
         म: 5,494
         .: 5,398
 

In [15]:
# Start training with individual arguments
train_manifest = config['data']['train_manifest']
val_manifest = config['data']['val_manifest']
vocab_file = config['data']['vocab_file']

!cd /kaggle/working && PYTHONPATH=/kaggle/working python training_scripts/train_konkanivani_asr.py \
    --train_manifest {train_manifest} \
    --val_manifest {val_manifest} \
    --vocab_file {vocab_file} \
    --batch_size {config['training']['batch_size']} \
    --num_epochs {config['training']['num_epochs']} \
    --learning_rate {config['training']['learning_rate']} \
    --weight_decay {config['training']['weight_decay']} \
    --dropout {config['model']['dropout']} \
    --ctc_weight {config['training']['ctc_weight']} \
    --save_every {config['training']['save_every']} \
    --checkpoint_dir {config['paths']['checkpoint_dir']} \
    --log_dir {config['paths']['log_dir']} \
    --d_model {config['model']['d_model']} \
    --encoder_layers {config['model']['encoder_layers']} \
    --decoder_layers {config['model']['decoder_layers']} \
    --gradient_accumulation_steps {config['training']['gradient_accumulation_steps']} \
    --mixed_precision \
    --device cuda

2025-12-15 03:31:14.168575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765769474.381892      72 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765769474.438974      72 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/tensorboard/compat/__init__.py", line 42, in tf
    from tensorboard.compat import notf  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: cannot import name 'notf' from 'tensorboard.compat' (/usr/local/lib/python3.11/dist-packages/tensorboard/compat/__init__.py)

During handling of the above exception, another exception occurred:

Attr

## Step 7: Monitor Progress

### Expected Timeline:
- **Epoch 1-10**: Blank prob 95-98% (learning basics)
- **Epoch 10-20**: Blank prob 80-90% (characters appearing)
- **Epoch 20-40**: Blank prob 50-80% ✅ **WORKING!**
- **Epoch 40-100**: Blank prob 30-50% (refinement)

In [16]:
# Check test results
test_results_dir = Path('/kaggle/working/checkpoints')
test_files = sorted(test_results_dir.glob('test_results_epoch_*.json'))

if test_files:
    print("Test Results Summary:")
    print("=" * 80)
    for test_file in test_files:
        with open(test_file) as f:
            results = json.load(f)
            epoch = results.get('epoch', '?')
            blank_prob = results.get('avg_blank_prob', 0)
            status = '✅ WORKING!' if blank_prob < 80 else '❌ Not yet'
            print(f"Epoch {epoch:3d}: Blank prob {blank_prob:5.1f}% - {status}")
else:
    print("No test results yet. Check back after epoch 5.")

No test results yet. Check back after epoch 5.


## Step 8: Download Best Checkpoint

In [17]:
# Find best checkpoint (lowest validation loss)
checkpoint_dir = Path('/kaggle/working/checkpoints')
checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))

if checkpoints:
    best_ckpt = None
    best_val_loss = float('inf')
    
    for ckpt_path in checkpoints:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_loss = ckpt.get('val_loss', float('inf'))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt = ckpt_path
    
    print(f"Best checkpoint: {best_ckpt.name}")
    print(f"Validation loss: {best_val_loss:.4f}")
    
    # Copy to best_model.pt
    import shutil
    shutil.copy(best_ckpt, checkpoint_dir / 'best_model.pt')
    print("✓ Saved as best_model.pt")
else:
    print("No checkpoints found yet")

Best checkpoint: checkpoint_epoch_99.pt
Validation loss: 2.0637
✓ Saved as best_model.pt


In [18]:
# Create download link
from IPython.display import FileLink

print("Download your trained model:")
FileLink('/kaggle/working/checkpoints/best_model.pt')

Download your trained model:


/kaggle/working/checkpoints/best_model.pt

## Step 9: Quick Test

In [19]:
# Test the best model on a few samples
!python /kaggle/working/scripts/test_best_model.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pt \
    --max_files 10

ASR MODEL INFERENCE TEST
Loading vocabulary from data/vocab.json...
Traceback (most recent call last):
  File "/kaggle/working/scripts/test_best_model.py", line 251, in <module>
    main()
  File "/kaggle/working/scripts/test_best_model.py", line 213, in main
    asr = ASRInference(args.checkpoint, args.vocab)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/scripts/test_best_model.py", line 28, in __init__
    with open(vocab_path, 'r', encoding='utf-8') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/vocab.json'


## Step 10: Generate Training Visualization

In [20]:
# Generate comprehensive training metrics visualization
!python /kaggle/working/scripts/generate_training_visualization.py \
    --log /kaggle/working/logs/training.log \
    --output /kaggle/working/training_metrics.png \
    --title "Konkani ASR"

✗ Log file not found: /kaggle/working/logs/training.log

Searching for log files...
✗ No log files found. Please specify --log path


In [21]:
# Display the visualization
from IPython.display import Image, display
import os

if os.path.exists('/kaggle/working/training_metrics.png'):
    print("✓ Training Visualization:")
    display(Image('/kaggle/working/training_metrics.png'))
else:
    print("✗ Visualization not generated yet. Run after training completes.")

✗ Visualization not generated yet. Run after training completes.


In [22]:
# Download link for the visualization
from IPython.display import FileLink

print("Download training visualization:")
FileLink('/kaggle/working/training_metrics.png')

Download training visualization:


/kaggle/working/training_metrics.png

## Summary

### Key Fixes Applied:
1. ✅ CTC weight: 0.3 → 0.8 (critical for transcription)
2. ✅ Learning rate: 0.0001 → 0.0003 (faster learning)
3. ✅ Added gradient clipping: 5.0 (stability)
4. ✅ Full dataset: 21h → 88h (4x more data)
5. ✅ Periodic testing: Monitor every 5 epochs

### Expected Results:
- Model should start working by epoch 20-30
- Blank probability should drop below 80%
- Transcriptions should be recognizable
- Final CER should be 20-40%

### Next Steps:
1. Download best_model.pt
2. Download training_metrics.png
3. Test locally on your audio files
4. Deploy for production use